# Life of a Token: A Transformer in Einstein Notation

Suppose we give a language model "small blue robot". Its immediate task is to
assign probabilities to possible continuations, such as "moved" or "stopped".
Choosing one continuation and repeating the process produces a longer passage.


## The Journey at a Glance

First, text becomes a sequence of numerical vectors. Attention lets each
position gather information from its context. A feed-forward network applies
learned transformations to each position. Together, these stages form a block;
successive blocks refine the representations. The final position supplies a
distribution over possible next tokens. Choosing a token extends the text,
ready for another trip through the model.

The model predicts a distribution of possibilities. A selection rule turns
that distribution into one token: we can choose its most probable candidate
or draw a sample. Appending the selected token creates the context for the next
prediction. We will follow both the prediction and the choice.

The examples use NumPy and einops (version 0.6 or newer). Run this setup once
in the notebook's kernel. All diagrams are generated inside the notebook.


In [ ]:
%pip install numpy "einops>=0.6"


In [ ]:
import numpy as np
from einops import einsum, rearrange, reduce, repeat


In [ ]:
# Visual geometry helpers
from IPython.display import SVG, display
from html import escape

COLORS = ['#58c4dd', '#ffcf56', '#fc9272', '#a7db8d']

def journey(active='all'):
    labels = ['I: Embeddings', 'II: Attention', 'III: Feed-forward', 'IV: Predict']
    links = ['part-i', 'part-ii', 'part-iii', 'part-iv']
    p = ['<svg xmlns="http://www.w3.org/2000/svg" width="960" height="225" viewBox="0 0 960 225" role="img" aria-label="Token journey and section navigation">',
         '<rect width="960" height="225" fill="#151923"/>']
    p.append('<rect x="250" y="20" width="450" height="110" rx="12" fill="none" stroke="#9da6b5" stroke-dasharray="5 4"/>')
    p.append('<text x="475" y="43" fill="white" text-anchor="middle" font-family="sans-serif" font-size="16">Transformer block: repeat with new parameters</text>')
    for i, (label, link) in enumerate(zip(labels, links)):
        x = 25 + i*235
        color = COLORS[i] if active in ('all', str(i+1)) else '#9299a5'
        p.append(f'<a href="#{link}"><rect x="{x}" y="60" width="205" height="50" rx="8" fill="#252d3b" stroke="{color}" stroke-width="2"/><text x="{x+102}" y="91" fill="{color}" text-anchor="middle" font-family="sans-serif" font-size="18">{label}</text></a>')
        if i < 3:
            p.append(f'<text x="{x+219}" y="91" fill="white" text-anchor="middle" font-size="22">→</text>')
    p.append('<path d="M 830 115 V 163 H 125 V 115" fill="none" stroke="#a7db8d" stroke-width="2"/>')
    p.append('<text x="475" y="191" fill="#a7db8d" text-anchor="middle" font-family="sans-serif" font-size="17">Choose a token → append it to the text → run again</text></svg>')
    display(SVG(''.join(p)))

class Geometry:
    """Equal-scale Cartesian panels, rendered as self-contained SVG."""
    def __init__(self, titles, bound=3):
        self.bound = bound
        self.parts = ['<svg xmlns="http://www.w3.org/2000/svg" width="960" height="410" viewBox="0 0 960 410" role="img">',
                      '<rect width="960" height="410" fill="#151923"/>',
                      '<defs><marker id="tip" markerWidth="7" markerHeight="7" refX="6" refY="3" orient="auto-start-reverse"><path d="M0,0 L6,3 L0,6" fill="context-stroke"/></marker></defs>']
        for panel, title in enumerate(titles):
            self.text(panel, (0, bound+0.5), title, '#ffffff', 18, anchor='middle')
            self.line(panel, (-bound,0), (bound,0), '#586174')
            self.line(panel, (0,-bound), (0,bound), '#586174')
            for tick in [-bound, bound]:
                self.text(panel, (tick,-0.18), f'{tick:g}', '#a0a8b5', 11)
                self.text(panel, (0.08,tick), f'{tick:g}', '#a0a8b5', 11)
    def point(self, panel, a):
        return (240+480*panel+140*a[0]/self.bound, 205-140*a[1]/self.bound)
    def line(self, panel, a, b, color, width=1, arrow=False, dashed=False):
        x,y = self.point(panel,a); u,v = self.point(panel,b)
        self.parts.append(f'<path d="M{x},{y} L{u},{v}" fill="none" stroke="{color}" stroke-width="{width}"'+(' marker-end="url(#tip)"' if arrow else '')+(' stroke-dasharray="5 4"' if dashed else '')+'/>')
    def path(self, panel, points, color, width=2):
        xy = [self.point(panel,p) for p in points]
        self.parts.append('<polyline points="'+' '.join(f'{x},{y}' for x,y in xy)+f'" fill="none" stroke="{color}" stroke-width="{width}"/>')
    def text(self, panel, a, label, color='#ffffff', size=13, anchor='start'):
        x,y = self.point(panel,a)
        self.parts.append(f'<text x="{x}" y="{y}" fill="{color}" font-family="sans-serif" font-size="{size}" text-anchor="{anchor}">{escape(label)}</text>')
    def dot(self, panel, a, color, radius=5):
        x,y = self.point(panel,a)
        self.parts.append(f'<circle cx="{x}" cy="{y}" r="{radius}" fill="{color}"/>')
    def vector(self, panel, a, color, label='', start=(0,0)):
        if np.linalg.norm(np.asarray(a)-np.asarray(start)) > 1e-12:
            self.line(panel,start,a,color,3,True)
        else:
            self.dot(panel,a,color)
        if label:
            self.text(panel, np.asarray(a)+[0.12,0.15], label, color)
    def caption(self, panel, label):
        self.parts.append(f'<text x="{240+480*panel}" y="382" fill="#d7deeb" font-family="sans-serif" font-size="14" text-anchor="middle">{escape(label)}</text>')
    def show(self):
        display(SVG(''.join(self.parts)+'</svg>'))

def candidate_picture(names, logits, probability, title):
    height = 100 + 46*len(names)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="960" height="{height}" viewBox="0 0 960 {height}" role="img">',
             f'<rect width="960" height="{height}" fill="#151923"/>']
    def label(x,y,text,color='#ffffff',size=16):
        parts.append(f'<text x="{x}" y="{y}" fill="{color}" font-family="sans-serif" font-size="{size}">{escape(str(text))}</text>')
    label(30,30,title,size=20)
    for x,name in [(30,'Candidate'),(240,'Logit'),(370,'Probability'),(520,'Share of probability mass')]:
        label(x,65,name,'#aeb8c8',14)
    for i,(name,score,p) in enumerate(zip(names,logits,probability)):
        y=100+46*i
        label(30,y,name)
        label(240,y,f'{score:.3f}')
        label(370,y,f'{p:.3f}')
        parts.append(f'<rect x="520" y="{y-17}" width="{400*p}" height="24" fill="{COLORS[i%4]}"/>')
    display(SVG(''.join(parts)+'</svg>'))


In [ ]:
# Rendering
journey()


Follow the four parts in order, or use the diagram to return to a stage:

- [Part I](#part-i) builds the geometric vocabulary for representing and transforming information.
- [Part II](#part-ii) explains communication between token positions.
- [Part III](#part-iii) explains feature recognition and learned associations.
- [Part IV](#part-iv) assembles the complete executable model.

The parts share definitions. On a fresh kernel, run the preceding cells before
starting a later part. Folded drawing cells still need to run; their outputs
show the geometry while the visible cells show the tensor computation.

For three lectures, a natural route is Part I, then Parts II–III, then Part IV.
The lab storage-layout subsection is an optional detour; the complete model
does not depend on its adapter or verification code.

<a id="part-i"></a>


## Part I: Understanding Embeddings and Tensor Primitives


In [ ]:
# Rendering
journey('1')


### Axis names: a map for reading the model

Each axis answers a question about an array. We use one short label per role
and keep its meaning consistent. This table is a reference; each part
introduces its axes alongside the operation that uses them.

| Label | Meaning: what one index selects |
|-------|--------------------------------|
| `m` | Model coordinate: one coordinate of a token's representation |
| `t` | Target position: the token position receiving information |
| `s` | Source position: the token position supplying information |
| `h` | Head: one attention head |
| `d` | Coordinate within a head's comparison or message vector |
| `f` | Feature: one feed-forward hidden feature |
| `v` | Vocabulary entry: one candidate token |

Lowercase labels identify axes; uppercase letters such as M, T, and H give
their sizes. Source and target positions range over the same sequence, with
different roles. In the small model, query/key vectors and value vectors
have the same width D. Their spaces serve different purposes: Q and K are
compared, while V is combined into messages. The short label d is reused
in those separate operations.

For example, `d m, m t -> d t` applies one learned map to every target position
and sums over model coordinates m. Likewise, `d s, d t -> s t` compares every source
with every target by summing over comparison coordinates d. The axes after
the arrow tell us exactly what remains.

Our vectors are columns: E has shape (M,T). Attention scores are K.T @ Q,
with sources down the rows and targets across the columns. Each score column
is normalized over its source positions. Keep that orientation when reading
both the pictures and the expressions.

These are our model-wide naming conventions. Each einsum call interprets
its labels locally; consistent names let us follow information between calls.
Geometry explains what happens to one vector, while the named axes explain
how that operation applies across positions, heads, and sequences.


### Text becomes vectors

The model first splits text into pieces called *tokens*: words, parts of words,
or punctuation. Its *vocabulary* is the collection of tokens it recognizes,
each identified by an integer ID. For our small examples, each word is one token.

The model computes with numerical vectors. A lookup table associates each
token ID with a vector called its *embedding*. The model repeatedly updates
these vectors using information from the preceding text. Finally, it uses the
last position's vector to score possible next tokens. We will follow that journey
and express its numerical operations in Einstein notation.

Token representations occupy columns in our pictures. A token's ID stays the
same as it passes through the model; its vector representation changes.

Training adjusts the numerical parameters using examples of text and the
tokens that follow it. During inference, these parameters stay fixed while
the intermediate representations change with the input. Our hand-selected
parameters make individual mechanisms visible; the lab supplies trained
parameters that support useful predictions.


### Reading an embedding array

Consider "small blue robot moved". The lookup for robot gives the same vector
as it would in "large red robot moved". Yet the surrounding words tell us about
different robots. We need a computation that brings information from small and
blue into the representation at robot. This selective transfer of information
between positions is called *attention*.

For this example we invent four interpretable coordinates: descriptor, object,
size information, and color information. This tiny hand-built model lets us
inspect each coordinate. Each token in this example corresponds to one word.
The embedding table and other numerical settings of a trained model are called
its *parameters*. Training adjusts them using examples; running the model uses
them as fixed numerical settings. Here we choose numbers by hand so their effect is
easy to inspect.

Write E[m,t] for coordinate m of the embedding at position t. Lowercase letters
inside an einsum expression label axes; uppercase M and T will denote their
sizes. Here M=4 coordinates and T=4 positions. Lookup selects vectors; it is
ordinary array indexing. The products and sums that follow use einsum.


In [ ]:
import numpy as np
from einops import einsum, rearrange, reduce, repeat
from IPython.display import SVG, display
from html import escape
np.set_printoptions(precision=3, suppress=True)
tokens = ['small', 'blue', 'robot', 'moved']
features = ['descriptor', 'object', 'size', 'color']
E = np.array([[1, 1, 0, 0],
              [0, 0, 1, 0],
              [1, 0, 0, 0],
              [0, 1, 0, 0]], dtype=float)


In [ ]:
def matrix_picture(a, title, rows, columns, parameters=False, attention=False):
    a = np.asarray(a)
    nr, nc = a.shape
    assert len(rows) == nr and len(columns) == nc
    left, top, cw, ch = 150, 110, 110, 54
    width = max(left + nc*cw + 35, 40 + 12*len(title), 440)
    height = top + nr*ch + 70
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             f'<rect width="{width}" height="{height}" fill="#10131a"/>']
    def text(x, y, label, color='#eeeeee', size=14, anchor='middle'):
        parts.append(f'<text x="{x}" y="{y}" fill="{color}" font-family="monospace" font-size="{size}" text-anchor="{anchor}">{escape(str(label))}</text>')
    text(20, 30, title, size=19, anchor='start')
    scale = max(float(np.max(np.abs(a[np.isfinite(a)]))), 1e-9)
    for j, label in enumerate(columns):
        text(left + j*cw + cw/2, top-30, label, '#f4d35e')
    for i, label in enumerate(rows):
        text(left-18, top+i*ch+ch/2+5, label, anchor='end')
        for j in range(nc):
            v = float(a[i,j]); x = left+j*cw+cw/2; y = top+i*ch+ch/2
            strength = abs(v)/scale if np.isfinite(v) else 0
            color = '#eeeeee'
            if parameters:
                base = (100,190,245) if v >= 0 else (245,110,120)
                color = '#'+''.join(f'{int(c*(0.45+0.55*strength)):02x}' for c in base)
            elif not attention:
                grey = int(130+110*strength)
                color = f'#{grey:02x}{grey:02x}{grey:02x}'
            if attention:
                radius = 21*np.sqrt(max(v,0)/scale)
                parts.append(f'<circle cx="{x}" cy="{y}" r="{radius}" fill="#245b78"/>')
            text(x,y+5, f'{v:.2f}' if np.isfinite(v) else '-inf', color)
    for x, direction in [(left-8, 1), (left+nc*cw+8, -1)]:
        parts.append(f'<path d="M {x+8*direction} {top-8} H {x} V {top+nr*ch+8} H {x+8*direction}" fill="none" stroke="#aaaaaa" stroke-width="2"/>')
    parts.append('</svg>')
    display(SVG(''.join(parts)))


In [ ]:
# Rendering
matrix_picture(E, 'E[m,t]: one column per token', features, tokens)


### Reading the tensor operations

A tensor is an array whose axes identify the quantities being organized.
In E[m,t], one axis identifies a coordinate and the other a token position.
An einsum expression tells us which axes remain in the result and which are
summed over.

For example, `'d m, m t -> d t'` applies a matrix to every token vector. For each
output coordinate d and position t, it multiplies the corresponding entries
and sums over m. The result retains the token positions and expresses each
vector in new coordinates. This operation is called a *linear projection*.

Three patterns will recur throughout the model:

- `'d m, m t -> d t'`: transform each token vector with shared parameters.
- `'d s, d t -> s t'`: compare every source vector with every receiving vector.
- `'d s, s t -> d t'`: combine source vectors using a weight for each source and receiver.

The index that disappears is summed over. The indices after the arrow tell us
what each entry of the result means. Attention gives these three patterns a
concrete purpose: form requests, compare them with sources, and gather information.

We use einops throughout: `einsum` expresses products and sums, `rearrange`
splits, combines, and permutes axes, `reduce` combines entries using sum,
mean, or maximum, and `repeat` shares data along a new axis.
Patterns separate axis names with spaces; operands precede the pattern:
`einsum(Wq, E, 'd m, m t -> d t')`.
Elementwise mathematics and indexing complete the computation.
Comments beside translated expressions preserve
their ordinary NumPy form; assertions and the loop reference use that form
to check the results independently.


### Lookup: select a learned vector

An embedding table stores one column per vocabulary entry. A sequence of token
IDs selects columns in sentence order; repeated IDs select the same column.
Here the table contains our four words and the prompt selects its first three.


In [ ]:
embedding_table = E.copy()
prompt_ids = np.array([0, 1, 2])
prompt_vectors = embedding_table[:, prompt_ids]


In [ ]:
# Rendering
matrix_picture(prompt_vectors, 'Lookup: small, blue, robot', features, tokens[:3])


Lookup connects discrete symbols with numerical computation. Training learns
the table's entries along with the rest of the model's parameters.


### Linear maps and broadcasting: detect and combine features

Imagine a square grid drawn on a sheet. A linear map moves every point
according to the same rule: straight grid lines stay straight, while their
directions and spacing change. A bias translates the transformed grid.
Follow the blue vector through the two panels; each plotted point is computed
by the same matrix used in the einsum below. The axes have equal scales.


In [ ]:
plane = np.array([[1.0, 0.6], [-0.3, 0.8]])
shift = np.array([0.4, 0.5])
geometry_probe = np.array([1.0, 1.0])
mapped_probe = einsum(plane, geometry_probe, 'i j, j -> i') + shift
grid_segments = np.array([segment for c in range(-2,3)
                          for segment in ([[c,-2],[c,2]], [[-2,c],[2,c]])])
# Original: segment @ plane.T + shift for each grid segment
mapped_segments = einsum(plane, grid_segments, 'i j, s p j -> s p i') + shift


In [ ]:
# Rendering
fig = Geometry(['Original coordinates', 'Linear map, then translation'])
for segment, mapped in zip(grid_segments, mapped_segments):
    fig.path(0,segment,'#424d62',1)
    fig.path(1,mapped,'#596680',1)
fig.vector(0,geometry_probe,COLORS[0],'x')
fig.vector(1,shift,COLORS[1],'bias')
fig.vector(1,mapped_probe,COLORS[0],'', start=shift)
fig.dot(1,mapped_probe,COLORS[0])
fig.text(1,mapped_probe+[0.12,0.15],'Wx + bias',COLORS[0])
fig.caption(0,'One point, two coordinates')
fig.caption(1,"einsum(W, x, 'i j, j -> i') + bias")
fig.show()


A linear map forms new features as weighted sums of existing coordinates.
Each row of the parameter matrix specifies one such combination. Applying
the same matrix at every position gives *parameter sharing* across tokens.

A *bias* adds an offset to each output coordinate. The singleton position axis
in `bias[:,None]` broadcasts that offset across all positions. Together the
matrix and bias form an *affine layer*, often called a dense layer.


In [ ]:
feature_map = np.array([[0, 0, 1, 1],
                        [0, 1, -1, -1]], dtype=float)
bias = np.array([0.25, 0.0])
detected = einsum(feature_map, E, 'f m, m t -> f t') + bias[:,None]
np.testing.assert_allclose(detected, feature_map @ E + bias[:,None])


In [ ]:
# Rendering
matrix_picture(detected, 'Shared affine map at every position',
               ['size + color + offset', 'object - size - color'], tokens)


Here f labels an output feature. The first feature collects descriptive
information; the second assigns opposite signs to objects and descriptions.
The same arithmetic handles all four tokens in one expression.


### Nonlinear activation and gating: make features interact

ReLU clips each coordinate at zero. Geometrically, points in the negative-x
half-plane move onto the vertical axis; points with negative y move onto the
horizontal axis. Points with both coordinates negative meet at the origin.
The resulting bends give compositions of layers their expressive power.


In [ ]:
activation_points = np.array([[-2,1], [1,-2], [-1,-1], [1,2]], dtype=float)


In [ ]:
# Rendering
fig = Geometry(['Before ReLU', 'After coordinate-wise ReLU'])
for i, point in enumerate(activation_points):
    clipped = np.maximum(point,0)
    fig.vector(0,point,COLORS[i],str(i+1))
    fig.vector(1,clipped,COLORS[i],str(i+1))
    fig.dot(1,clipped,COLORS[i])
fig.caption(0,'The same four inputs')
fig.caption(1,'maximum(x, 0) acts on each coordinate')
fig.show()


An *activation function* acts on each entry of an array. For example, ReLU
keeps positive entries and maps negative entries to zero. A second linear
map can then combine the features selected by this nonlinear operation.
This composition lets a network express richer functions than a single affine map.

The lab uses *SiLU*, defined by `u / (1 + exp(-u))`. Its smooth response
scales each entry according to its value. A *gate* multiplies one computed
array by another of the same shape, letting one feature modulate another.


In [ ]:
relu_features = np.maximum(detected, 0)
silu_features = detected / (1 + np.exp(-detected))
gate_features = np.array([[1, 2, 1, 1], [1, 1, 0.5, 1]])
gated_features = silu_features * gate_features


In [ ]:
# Rendering
matrix_picture(gated_features, 'SiLU followed by elementwise gating',
               ['feature 0', 'feature 1'], tokens)


The product keeps both axes. Each entry interacts with the entry at the same
feature and position. In the full model, a learned projection produces the gate.


### Reduction and normalization: control a vector's scale

In two dimensions, RMS normalization moves each nonzero vector along its ray
toward a circle of radius sqrt(2). A learned scale for each coordinate then
reshapes the circle into an ellipse. The picture uses epsilon=0 to display
the exact circle; the numerical model uses a small positive epsilon.


In [ ]:
norm_inputs = np.array([[2.4,0.8], [0.4,1.0], [-1.8,1.6]])
# Original: np.mean(norm_inputs**2, axis=1, keepdims=True)
# The local axis n enumerates geometric examples.
norm_mean_square = reduce(norm_inputs**2, 'n m -> n 1', 'mean')
norm_outputs = norm_inputs / np.sqrt(norm_mean_square)
learned_scale = np.array([1.5,0.65])
angles = np.linspace(0,2*np.pi,160)
# Original: np.array([np.cos(angles),np.sin(angles)]).T
circle = np.sqrt(2)*rearrange(np.array([np.cos(angles),np.sin(angles)]), 'm n -> n m')


In [ ]:
# Rendering
fig = Geometry(['Normalize along each ray', 'Learned coordinate scales'])
fig.path(0,circle,'#8892a4')
fig.path(1,circle*learned_scale,'#8892a4')
for i,(before,after) in enumerate(zip(norm_inputs,norm_outputs)):
    fig.vector(0,before,'#697184')
    fig.vector(0,after,COLORS[i])
    fig.line(0,before,after,COLORS[i],2,True,True)
    fig.vector(1,after*learned_scale,COLORS[i])
fig.caption(0,'Direction stays; RMS magnitude becomes one')
fig.caption(1,'Scale x by 1.5 and y by 0.65')
fig.show()


A *reduction* combines entries along an axis. Summing over m in
`'m t -> t'` produces one number per token. A mean divides that sum by the
number of coordinates. Keeping a singleton coordinate axis makes the result
ready to broadcast back across each column.

*Root-mean-square normalization* divides each token vector by its RMS magnitude
and then applies a learned scale per coordinate. The small positive epsilon
keeps the denominator positive. The model uses this operation to control the
scale of vectors entering its learned transformations.


In [ ]:
# Original: np.mean(E * E, axis=0)
mean_square = reduce(E * E, 'm t -> t', 'mean')
rms = np.sqrt(mean_square + 1e-5)
coordinate_scale = np.ones(E.shape[0])
scaled_embeddings = coordinate_scale[:,None] * E / rms[None,:]
np.testing.assert_allclose(mean_square, np.mean(E * E, axis=0))


In [ ]:
# Rendering
matrix_picture(scaled_embeddings, 'RMS normalization: scale each column', features, tokens)


### Softmax: turn scores into shares

Softmax turns scores into positive weights that sum to one:
`p[i] ` exp(score[i]) / sum(exp(score[j]) for j in candidates)=.
Subtracting the largest score before exponentiation preserves these weights
and avoids unnecessarily large exponentials.

Three nonnegative shares summing to one describe a point inside a triangle.
Each corner assigns the whole share to one candidate. The centre assigns
equal shares. Softmax maps a score vector to a point in this /probability
simplex/. As one score grows relative to the others, the point approaches
that candidate's corner.

*Predict before running:* Add 10 to all three scores. Will the point in the
probability triangle move? Explain your prediction using the softmax formula.


In [ ]:
def shares(scores):
    # Original: scores - np.max(scores)
    exp_scores = np.exp(scores - reduce(scores, 'v ->', 'max'))
    # Original: exp_scores / exp_scores.sum()
    return exp_scores / reduce(exp_scores, 'v -> ', "sum")
simplex_corners = np.array([[-1.5,-1], [1.5,-1], [0,1.6]])
value_corners = np.array([[0.2,1.8], [-1.6,-0.7], [1.8,-0.7]])
score_path = [np.array([strength,0.,0.]) for strength in [0,0.7,1.5,3.]]
share_path = np.array([shares(score) for score in score_path])
# Original: share_path @ corners, for each of the two coordinate systems
# Here n enumerates score settings, and d is a plotted coordinate.
mixture_paths = [einsum(share_path, corners, 'n v, v d -> n d')
                 for corners in [simplex_corners, value_corners]]
share_path


Adding the same constant multiplies every exponential by the same factor,
which cancels in the normalization. The point stays in the same place.


In [ ]:
# Rendering
fig = Geometry(['Scores become mixture weights', 'The weights combine value vectors'])
for panel,corners in enumerate([simplex_corners,value_corners]):
    fig.path(panel,np.vstack([corners,corners[0]]),'#8892a4')
    for i,corner in enumerate(corners):
        fig.dot(panel,corner,COLORS[i])
        fig.text(panel,corner+[0.08,0.12],str(i+1),COLORS[i])
    positions = mixture_paths[panel]
    fig.path(panel,positions,'#ffffff')
    for i,position in enumerate(positions):
        fig.dot(panel,position,'#ffffff',3+i)
    fig.text(panel,positions[0]+[0.1,-0.35],'equal shares')
fig.caption(0,'Increase score 1: 0 → 0.7 → 1.5 → 3')
fig.caption(1,"einsum(values, weights, 's d, s -> d')")
fig.show()


The two panels use exactly the same weights. The left triangle locates the
probability distribution; the right locates the resulting mixture of three
value vectors. This link will explain how attention scores produce an update.

Suppose three candidates receive scores 2, 1, and 0. We want larger scores
to receive larger shares, with all shares positive and their sum equal to one.
*Softmax* exponentiates the scores and divides by their sum.

Subtracting the largest score first gives exactly the same shares while
keeping the exponentials within a convenient numerical range.


In [ ]:
candidate_scores = np.array([2.0, 1.0, 0.0])
# Original: candidate_scores - candidate_scores.max()
candidate_weights = np.exp(candidate_scores - reduce(candidate_scores, 'v ->', 'max'))
# Original: candidate_weights / candidate_weights.sum()
candidate_weights = candidate_weights / reduce(candidate_weights, 'v -> ', "sum")
np.testing.assert_allclose(candidate_weights.sum(), 1)
candidate_weights


The result is approximately [0.665, 0.245, 0.090]. Attention will use these
shares to combine source vectors. At the end of the model, softmax will turn
vocabulary scores into probabilities for the next token. Choosing the axis
identifies which candidates compete for the total share of one.


### From a representation to candidate tokens

To finish a prediction, we need one score for each entry in the vocabulary.
An *output projection*, also called an *unembedding matrix*, has one row per
candidate. Each row is a learned scoring direction. Its dot product with the
final representation gives that candidate's *logit*: a raw score that can be
positive, zero, or negative. Softmax turns the logits into probabilities.

Here is the geometry in a two-dimensional model with three candidate tokens.
The representation is the white vector. The three scoring directions produce
logits 2, 1, and 0. These are illustrative choices of coordinates and parameters.


In [ ]:
output_example = np.array([1.0, 0.5])
output_directions = np.array([[2.,0.], [0.,2.], [-0.5,1.]])
output_candidates = ['moved', 'stopped', 'today']
output_scores = einsum(output_directions, output_example, 'v m, m -> v')
def token_probabilities(logits, temperature=1.0):
    # A positive temperature scales the logits before softmax.
    assert temperature > 0
    # Original: (logits - np.max(logits)) / temperature
    scaled = (logits - reduce(logits, 'v ->', 'max')) / temperature
    weights = np.exp(scaled)
    # Original: weights / weights.sum()
    return weights / reduce(weights, 'v -> ', "sum")
output_probabilities = token_probabilities(output_scores)
temperature_values = [2.,1.,0.5]
# Original: token_probabilities(output_scores,temp) @ simplex_corners
temperature_points = np.array([einsum(token_probabilities(output_scores,temp), simplex_corners, 'v, v d -> d')
                               for temp in temperature_values])


In [ ]:
# Rendering
fig = Geometry(['Candidate scoring directions', 'Softmax distribution and temperature'])
fig.vector(0,output_example,'#ffffff','representation')
for i,direction in enumerate(output_directions):
    fig.vector(0,direction,COLORS[i],output_candidates[i])
fig.path(1,np.vstack([simplex_corners,simplex_corners[0]]),'#8892a4')
for i,corner in enumerate(simplex_corners):
    fig.dot(1,corner,COLORS[i])
    offset = [-0.9,0.2] if i == 0 else [0.08,0.15]
    fig.text(1,corner+offset,output_candidates[i],COLORS[i])
fig.path(1,temperature_points,'#ffffff')
for temp,point,offset in zip(temperature_values,temperature_points,
                             [[0.2,0.45],[-0.1,-0.8],[-1.0,-0.45]]):
    fig.dot(1,point,'#ffffff')
    label_position = point + offset
    fig.line(1,point,label_position,'#8892a4',1)
    fig.text(1,label_position,f'temp {temp:g}')
fig.caption(0,"einsum(W, x, 'v m, m -> v'): one logit per token")
fig.caption(1,'Temperature 2 → 1 → 0.5: concentrate toward moved')
fig.show()
candidate_picture(output_candidates,output_scores,output_probabilities,
                  'The same three candidates: logits become probabilities')


Here v remains in the result and m is summed over. For all positions at once,
the expression becomes `v m, m t -> v t`. The output matrix is learned alongside
the embedding table and the rest of the model.

*Temperature* divides logits by a positive number before softmax. Lower
temperatures concentrate probability on higher-scoring candidates; higher
temperatures spread it more evenly. Their ranking stays the same. In the
triangle, the distribution moves toward the highest-scoring candidate as
temperature decreases. Temperature changes the distribution used for a choice;
the model's parameters remain fixed.


### Predicting a distribution and choosing a token

*Greedy decoding* chooses the largest probability. *Sampling* draws a token
according to the distribution: a candidate with probability 0.245 has a 24.5%
chance on each draw from that distribution. A seed makes our demonstration
repeatable. Repeated draws illustrate the choice at one fixed context.


In [ ]:
greedy_example = int(np.argmax(output_probabilities))
selection_rng = np.random.default_rng(326)
sampled_examples = selection_rng.choice(len(output_candidates),size=12,p=output_probabilities)
print('Greedy:',output_candidates[greedy_example])
print('Samples at the same context:',[output_candidates[i] for i in sampled_examples])


A sample is one discrete outcome represented by a corner of the probability
triangle; the distribution is the interior point describing the chances.
In text generation, each selected token is appended to the input. The next
pass computes a new distribution for that extended context. Generation ends
when a designated end-of-sequence token is chosen or a length limit is reached.
Our miniature model uses a length limit; Part IV will run the complete loop.


### Weighted combination and residual addition: gather and refine

A weighted combination multiplies each source vector by its share and adds
the results. The source index s disappears; the vector's coordinate index m
remains. Using the first three embedding columns makes the effect visible.
Here we reuse the illustrative shares [0.665, 0.245, 0.090] as weights for
the source positions small, blue, and robot. These weights now describe how
much each source contributes to a mixture, rather than next-token probabilities.


In [ ]:
mixture = einsum(prompt_vectors, candidate_weights, 'm s, s -> m')
np.testing.assert_allclose(mixture, prompt_vectors @ candidate_weights)


In [ ]:
# Rendering
matrix_picture(mixture[:,None], 'Weighted combination of three source vectors',
               features, ['mixture'])


For several receivers, we arrange the shares in columns and write
`'m s, s t -> m t'`. Each receiver gets its own mixture of source information.

A *residual addition* adds an update to the current representation:
`refined ` current + update=. Both arrays have the same shape. This gives
each stage a way to refine a representation while carrying the existing
information forward. Attention and the feed-forward network both use it.


### More axes: repeat the same computation

An extra index can organize several versions of an operation. For example,
`'h f m, m t -> h f t'` applies h different feature maps to the same sequence.
An attention *head* will be one such separately parameterized computation.
A batch index b selects an independent input sequence: `'f m, b m t -> b f t'` applies
the same map to each sequence and each position.

The vocabulary is now in place: lookup, linear maps, broadcasting, activation,
gating, reduction, normalization, softmax, weighted combination, and residual
addition. Part II composes these operations into communication between tokens;
Part III develops learned associations, and Part IV assembles the complete model.

<a id="part-ii"></a>


## Part II: Understanding Attention


In [ ]:
# Rendering
journey('2')


We will follow the representation of robot as it gathers size and color
information from the preceding words. Each stage answers one question:
what information is needed, where it comes from, and how much to gather.


### What is robot looking for, and who can supply it?

Imagine robot looking for descriptions, while small and blue advertise that
they are descriptions. We need two numerical roles. A *query* is a vector
describing what a receiving position seeks. A *key* is a vector used to match
a source position to that request. Both roles are encoded numerically, and a
dot product measures their compatibility.

Each position produces both a query and a key from its current representation.
A matrix multiplication makes each new vector. Such a learned linear mapping
is commonly called a *projection* here: a general linear map between vector spaces.
Wq produces queries and Wk produces keys. They are model parameters, shared
across positions. The attention weights calculated later depend on the sentence.

Let t label a receiving position, s a source position, and d a query/key
coordinate. The labels s and t range over the same sequence but have different
roles. Our query/key width is D=2, while the embedding width is M=4.

For this constructed example, Wq turns the object coordinate into a request,
and Wk turns the descriptor coordinate into a matching advertisement. The second
coordinate distinguishes size from color; robot's query has a slight preference
for size. Both coordinates are visible in the geometric comparison below.
In a trained model, information can be distributed across many coordinates.


In [ ]:
Wq = np.array([[0, 2, 0, 0], [0, 0.3, 0, 0]], dtype=float)
Wk = np.array([[2, 0, 0, 0], [0, 0, 1, -1]], dtype=float)
Q = einsum(Wq, E, 'd m, m t -> d t')
K = einsum(Wk, E, 'd m, m s -> d s')
assert Q.shape == K.shape == (2, 4)
np.testing.assert_allclose(Q, Wq @ E)


In [ ]:
# Rendering
matrix_picture(Wq, 'Wq[d,m]: learned parameter layout (hand-set here)', ['d0','d1'], features, parameters=True)
matrix_picture(Q, 'Q[d,t]: queries', ['d0','d1'], tokens)
matrix_picture(K, 'K[d,s]: keys', ['d0','d1'], tokens)


For one output, Q[d,t] = sum over m of Wq[d,m] * E[m,t].
The m label disappears: we reduce features. The t label remains: we apply
the same projection independently at every token position.

Check yourself: changing Wq changes the rule for every token; changing one
column of E changes the input at one position. Parameters and input data play
different roles even though both are arrays.


### Which sources match robot's request?

The query supplies a direction along which to measure each key. Projecting
a key onto the query's direction gives a signed length; multiplying by the
query's length gives their dot product. Here small and blue have equally long
keys, so their different alignments explain their different scores.


In [ ]:
robot_query = Q[:,2]
# Original: robot_query / np.linalg.norm(robot_query)
query_direction = robot_query / np.sqrt(einsum(robot_query, robot_query, 'd, d -> '))
# Original: np.dot(key, query_direction) * query_direction for each key
key_projections = einsum(K, query_direction, 'd s, d -> s')[:,None] * query_direction


In [ ]:
# Rendering
fig = Geometry(['Robot query and source keys', 'Measure keys along the query'])
fig.vector(0,robot_query,COLORS[2],'query')
fig.line(1,-2.5*query_direction,2.5*query_direction,'#fc9272',2)
for i in range(2):
    key = K[:,i]
    projection = key_projections[i]
    fig.vector(0,key,COLORS[i],tokens[i])
    fig.vector(1,key,COLORS[i])
    fig.line(1,key,projection,COLORS[i],2,False,True)
    fig.dot(1,projection,COLORS[i])
fig.caption(0,'Key length and alignment determine compatibility')
fig.caption(1,'Signed projected length × query length')
fig.show()


For one receiving position t, compare its query with the key at each source
position s. A dot product multiplies corresponding coordinates and adds them:
score[s,t] = sum over d of K[d,s] * Q[d,t]. A larger score gives that source
more influence in the next step.

Doing this for every pair gives a grid: sources down the rows, receivers across
the columns. The whole grid is one einsum expression.


In [ ]:
scores = einsum(K, Q, 'd s, d t -> s t')
np.testing.assert_allclose(scores, K.T @ Q)
assert scores[0,2] == sum(K[d,0]*Q[d,2] for d in range(2))


In [ ]:
# Rendering
matrix_picture(scores, 'Scores[s,t]: source rows, receiving columns', tokens, tokens)


Read the robot column. Small has score 4.3 and blue has score 3.7. We reduced
d, but retained both s and t. The distinct labels
s and t ask for all pairs of positions in one expression.

The dot product measures compatibility through both direction and magnitude.
The next step turns these scores into normalized weights.


### How much should each source contribute?

A score can be negative or larger than one. To combine messages, we want
nonnegative *attention weights* that sum to one for each receiver. *Softmax*
does this by exponentiating each score and dividing by the sum of those
exponentials. Each allowed source receives a share, with larger scores receiving
larger shares. These attention weights depend on the input, while the learned
parameters Wq and Wk remain fixed during evaluation.

There is also a restriction. The representation at robot is used to predict
what follows robot, using the text available up to robot. Restricting information
to the current and earlier positions is called *causal attention*. A *mask*
excludes forbidden sources: set their scores to negative infinity before
softmax, so their exponentials and final weights become zero.

At receiving position t, only source positions s <= t are allowed. With sources
in rows, this is the upper triangle, including the diagonal. Softmax operates
down each column, summing over s. P[s,t] will denote the resulting weight.

Before softmax, divide scores by sqrt(D). Under a rough independent-unit-variance
model, dot product variance grows with the query/key width D; scaling counteracts
that growth. Subtracting a column maximum inside softmax has a separate purpose:
keeping exponentials numerically manageable while preserving the normalized result.

*Predict before running:* In the column for robot, which source must have
weight zero? In the column for the first token, small, what must its own
weight be? Identify the entries before looking at the matrix below.


In [ ]:
def softmax_columns(a):
    # Original: a - np.max(a, axis=0, keepdims=True)
    shifted = a - reduce(a, 's t -> 1 t', 'max')
    numerator = np.exp(shifted)
    # Original: numerator / np.sum(numerator, axis=0, keepdims=True)
    return numerator / reduce(numerator, 's t -> 1 t', 'sum')
T = len(tokens)
allowed = np.arange(T)[:,None] <= np.arange(T)[None,:]
masked = np.where(allowed, scores / np.sqrt(Q.shape[0]), -np.inf)
P = softmax_columns(masked)
np.testing.assert_allclose(P.sum(axis=0), np.ones(T))
assert np.all(P[~allowed] == 0)


In [ ]:
# Rendering
matrix_picture(masked, 'Masked scores: current and earlier sources', tokens, tokens)
matrix_picture(P, 'P[s,t]: each column sums to one', tokens, tokens, attention=True)


Every column includes its diagonal, so each receiver has at least one available
source and a positive softmax denominator.

Robot cannot receive information from moved, so that entry is zero. Small
has only itself as an available source, so its own weight is one.

The robot assigns most weight to the two descriptors. The remaining allowed
entry receives a small positive weight: softmax distributes influence smoothly.
The moved query is zero, so its distribution is uniform over its allowed prefix.
Our hand-set matching rule illustrates descriptor-to-object information transfer.


### Values: what information gets delivered?

We now know how strongly robot should draw on each source. The key helped
identify a source. We also need the information that source will deliver.
A source's *value* is a separate vector containing that information. Wv maps
each source representation to its value vector. Here it extracts size and color.

For robot, multiply each value vector by its attention weight and add the
results. C[d,t] is this collected message. In this example values have D
coordinates too. Keys support comparison; values supply the information we combine.
The *attention output projection* Wo maps the message back to M embedding
coordinates, producing delta[m,t]. Only then can it be added to E[m,t].
The final vocabulary projection instead maps a representation to one logit
per candidate token.

Adding an update to the incoming representation
is called a *residual addition*: updated = E + delta. This preserves a direct
path for the incoming information through the computation.

*Predict before running:* Exchange the size and color entries in Wv while
keeping the input, Wq, Wk, and Wo fixed. Which changes: the attention weights,
the delivered messages, or both?


In [ ]:
Wv = np.array([[0,0,1,0], [0,0,0,1]], dtype=float)
Wo = np.array([[0,0], [0,0], [1,0], [0,1]], dtype=float)
V = einsum(Wv, E, 'd m, m s -> d s')
C = einsum(V, P, 'd s, s t -> d t')
delta = einsum(Wo, C, 'm d, d t -> m t')
updated = E + delta
np.testing.assert_allclose(C[:,2], sum(P[s,2]*V[:,s] for s in range(T)))
np.testing.assert_allclose(delta, (Wo @ Wv) @ E @ P)


In [ ]:
# Rendering
matrix_picture(V, 'V[d,s]: the messages available at each source', ['size','color'], tokens)
matrix_picture(C, 'C[d,t]: weighted messages received', ['size','color'], tokens)
matrix_picture(updated, 'E + delta: robot now carries descriptor information', features, tokens)


In [ ]:
first_contribution = P[0,2]*V[:,0]


In [ ]:
# Rendering
fig = Geometry(['Robot gathers value vectors', 'Write the update into the embedding'], bound=1.5)
for i in range(3):
    fig.vector(0,V[:,i],COLORS[i],tokens[i])
fig.path(0,[V[:,0],V[:,1],V[:,2],V[:,0]],'#697184')
fig.vector(0,C[:,2],'#ffffff','mixture')
fig.vector(0,first_contribution,COLORS[0])
fig.vector(0,C[:,2],COLORS[1],start=first_contribution)
fig.vector(1,updated[2:,2],'#ffffff','updated robot',start=E[2:,2])
fig.dot(1,E[2:,2],COLORS[2])
fig.text(1,(-0.2,-0.3),'original robot')
fig.caption(0,f'small {P[0,2]:.3f}; blue {P[1,2]:.3f}; robot {P[2,2]:.3f}')
fig.caption(1,'Size/color plane of the 4D embedding')
fig.show()


The left panel draws the actual two-dimensional value space. The white vector
is the mixture; the colored arrows add the weighted size and color messages
head to tail. Robot's own value is zero. Wo places this result into the size
and color coordinates of the four-dimensional embedding. The right panel
shows those two coordinates; the object coordinate remains one.

The score calculation sums over feature coordinates to compare queries and
keys. The weighted value combination sums over source positions to collect
messages. Those different disappearing
indices explain the two different jobs.

The residual addition preserves the original representation while adding a
context-dependent update. Here we constructed an interpretable update; in a
trained network information is typically distributed across coordinates.

Changing Wv leaves the attention weights unchanged, but exchanges the size
and color components of the delivered messages. This is why keys and values differ.


### Several heads, several relationships

The query-key comparison followed by value collection is one *attention head*.
Different matching rules can capture different relationships. Multiple heads
run several such computations in parallel, with different parameter matrices;
the output projection combines their results into one update per position.

Introduce h for the head index and H for the number of heads. Each head has its
own projections and attention weights. The random parameters below demonstrate
the general computation with reproducible numerical inputs.


In [ ]:
rng = np.random.default_rng(326)
H, D, M = 2, 2, E.shape[0]
Wqh = rng.normal(size=(H,D,M))
Wkh = rng.normal(size=(H,D,M))
Wvh = rng.normal(size=(H,D,M))
Woh = rng.normal(size=(M,H,D))
Qh = einsum(Wqh, E, 'h d m, m t -> h d t')
Kh = einsum(Wkh, E, 'h d m, m s -> h d s')
Vh = einsum(Wvh, E, 'h d m, m s -> h d s')
Sh = einsum(Kh, Qh, 'h d s, h d t -> h s t') / np.sqrt(D)
Sh = np.where(allowed[None,:,:], Sh, -np.inf)
# Original: Sh - Sh.max(axis=1, keepdims=True)
exp_scores = np.exp(Sh - reduce(Sh, 'h s t -> h 1 t', 'max'))
# Original: exp_scores / exp_scores.sum(axis=1, keepdims=True)
Ph = exp_scores / reduce(exp_scores, 'h s t -> h 1 t', 'sum')
Ch = einsum(Vh, Ph, 'h d s, h s t -> h d t')
delta_h = einsum(Woh, Ch, 'm h d, h d t -> m t')
np.testing.assert_allclose(delta_h, Woh.reshape(M,H*D) @ Ch.reshape(H*D,T))
for h in range(H):
    np.testing.assert_allclose(Ch[h], Vh[h] @ Ph[h])


In [ ]:
# Rendering
for h in range(H):
    matrix_picture(Ph[h], f'Head {h}: a separate attention pattern', tokens, tokens, attention=True)


Here normalization reduces over source positions s, axis 1 in this layout. The
semantic rule remains the same: normalize over source positions for each query
in each head. Naming the axis is safer than remembering a numeric axis argument.

The output comparison above links directly to rearrange: flattening h,d and
multiplying is equivalent to reducing both named indices. Each head contributes
through its own part of the output projection.


### Optional lab detour: one query position and shared key/value heads

Our lab is a Llama implementation. Its attention kernel receives one query
position, a prefix of keys and values, and flattened heads. That prefix contains
exactly the available current and earlier positions. Selecting the prefix
therefore enforces the same causal rule as the triangular mask.

Some query heads share a key/value (KV) head. Let G be the number of KV heads.
If H=4 and G=2, the mapping is
[0,0,1,1]. This is grouped-query attention. Equal query and KV head counts are
the ordinary multi-head special case. The index selection below expands shared
heads explicitly so we can inspect the correspondence between heads.

Here g labels a shared KV head and h labels a query head. The single-query
adapter drops the query-position axis and places heads last: q[d,h] and
k[d,s,h]. Scores then have shape (S,H), so our column-softmax helper again
normalizes over sources s. The complete model returns to the heads-first
layout h,d,t for processing all query positions together.

Follow one concrete layout: three available positions, four query heads,
two shared KV heads, and two coordinates per head.

| Step | Query | Keys and values | Meaning |
|------|-------|-----------------|---------|
| Lab storage | (8,) | (3,4) | Heads are packed into each row. |
| Separate heads | (4,2) | (3,2,2) | Read the axes as h,d and s,g,d. |
| Share KV heads | (4,2) | (3,4,2) | Query heads 0/1 use KV head 0; heads 2/3 use KV head 1. |
| Arrange for einsum | (2,4) | (2,3,4) | Read the axes as d,h and d,s,h. |

The comparison `d s h, d h -> s h` gives three source scores for each of four
query heads. After softmax over s, `d s h, s h -> d h` gathers a two-coordinate
message for each head. Transposing and flattening returns eight coordinates
in the lab's original head order. The attention computation is the same;
these reshapes connect it to the lab's storage layout.


In [ ]:
def lab_attention_einsum(query, keys, values, position, heads, kv_heads):
    width = query.size // heads
    steps = position + 1
    # Original: query.reshape(heads, width).T
    q = rearrange(query, '(h d) -> d h', h=heads)
    # Original: keys[:steps].reshape(steps, kv_heads, width), likewise for values
    k = rearrange(keys[:steps], 's (g d) -> s g d', g=kv_heads)
    v = rearrange(values[:steps], 's (g d) -> s g d', g=kv_heads)
    head_map = np.arange(heads) // (heads // kv_heads)
    # Original: k[:,head_map,:].transpose(2,0,1), likewise for v
    k = rearrange(k[:,head_map,:], 's h d -> d s h')
    v = rearrange(v[:,head_map,:], 's h d -> d s h')
    score = einsum(k, q, 'd s h, d h -> s h') / np.float32(np.sqrt(width))
    probability = softmax_columns(score)             # sum over s
    context = einsum(v, probability, 'd s h, s h -> d h')
    # Original: context.T.reshape(-1).astype(query.dtype)
    return rearrange(context, 'd h -> (h d)').astype(query.dtype)


The adapter above is the implementation. The following independent check
compares it with the lab-style loop for several prefixes and head layouts.
Run it to verify the result; the next part can be read without tracing it.


In [ ]:
# Verification
# Independent loop-based reference, mirroring the lab's per-head calculation.
def lab_attention_loop(query, keys, values, position, heads, kv_heads):
    width = query.size // heads
    result = np.empty_like(query)
    for h in range(heads):
        g = h // (heads // kv_heads)
        q = query[h*width:(h+1)*width]
        k = keys[:position+1, g*width:(g+1)*width]
        v = values[:position+1, g*width:(g+1)*width]
        score = (k @ q) / np.float32(np.sqrt(width))
        p = np.exp(score - score.max())
        p /= p.sum()
        result[h*width:(h+1)*width] = p @ v
    return result
for heads, kv_heads in [(4,4), (4,2), (4,1)]:
    query = rng.normal(size=heads*6).astype(np.float32)
    keys = rng.normal(size=(7,kv_heads*6)).astype(np.float32)
    values = rng.normal(size=keys.shape).astype(np.float32)
    for position in range(7):
        actual = lab_attention_einsum(query,keys,values,position,heads,kv_heads)
        expected = lab_attention_loop(query,keys,values,position,heads,kv_heads)
        np.testing.assert_allclose(actual, expected, rtol=2e-5, atol=2e-6)
print('21 prefix/head-layout checks passed')


In the lab, *rotary positional embedding* (RoPE) supplies position information
by rotating adjacent query/key coordinate pairs before attention.
It changes the scores through Q and K while preserving the einsum pattern.
The lab then applies its output projection and adds the result to the residual.
RoPE, attention, and the residual update form successive steps in the model.

<a id="part-iii"></a>


## Part III: Understanding Feed-Forward Networks and Memory


In [ ]:
# Rendering
journey('3')


### Recognize a pattern, activate an association, write an update

Suppose context has brought together features for Eiffel and tower at one
position. A learned association could enrich that representation with Paris
and France. A feed-forward network supplies a mechanism for doing this:
its first matrix detects a pattern, its activation selects a response, and
its second matrix supplies a direction to write into the representation.

This gives attention and the feed-forward network complementary jobs.
Attention can bring the Eiffel clue to the tower position. The feed-forward
network can then recognize the combined clues there and add an association
encoded in its parameters. We will construct that second step explicitly.

Use four coordinates named
Eiffel, tower, Paris, and France. The three input columns contain both clues,
Eiffel alone, and tower alone. The detector adds the first two coordinates
and subtracts 1.5. ReLU gives a positive response when both binary clues are
present. In the input plane, the line x + y = 1.5 separates active inputs
from those whose response is zero.


In [ ]:
memory_inputs = np.array([[1,1,0], [1,0,1], [0,0,0], [0,0,0]], dtype=float)
memory_read = np.array([[1,1,0,0]], dtype=float)
memory_bias = np.array([-1.5])
memory_write = np.array([[0],[0],[2],[2]], dtype=float)
memory_features = np.maximum(einsum(memory_read, memory_inputs, 'f m, m t -> f t')
                             + memory_bias[:,None],0)
memory_update = einsum(memory_write, memory_features, 'm f, f t -> m t')
memory_result = memory_inputs + memory_update
memory_result


In [ ]:
# Rendering
fig = Geometry(['Recognize Eiffel + tower', 'Write Paris + France'], bound=2)
fig.line(0,(-0.5,2),(2,-0.5),'#ffffff',2,False,True)
fig.vector(0,[1,1],'#697184','detector direction')
for i, label in enumerate(['both','Eiffel','tower']):
    fig.dot(0,memory_inputs[:2,i],COLORS[i])
    fig.text(0,memory_inputs[:2,i]+[0.1,-0.2],label,COLORS[i])
fig.vector(1,memory_write[2:,0],COLORS[1],'write direction')
fig.vector(1,memory_update[2:,0],COLORS[0],'response × direction')
fig.caption(0,'Input plane: Eiffel (x), tower (y)')
fig.caption(1,'Output plane: Paris (x), France (y)')
fig.show()


For both clues, the hidden activation is 0.5 and the update is [0,0,1,1].
Each single-clue input has activation zero. The first matrix reads a feature;
the second writes an association. With many hidden features, the output is
a sum of such writes. The same matrices operate independently at every
token position.

The numbers here are hand-selected to expose a possible memory mechanism.
Training can adjust the detector, threshold, and write direction from examples.
Real models distribute associations across features and layers; individual
coordinates can participate in several patterns. This is an interpretation
of how parameterized computations can store associations, illustrated by a
small, fully inspectable construction.


### The lab's gated feed-forward network


After attention transfers information between positions, a /feed-forward
network/ (FFN) transforms each position's vector independently using the same
parameters. The intermediate vectors computed from an input are also called
*activations*; unlike model parameters, they change with the input.

Write these vectors as X[m,t]. *Root-mean-square normalization* (RMS normalization)
divides each column by sqrt(mean of its squared coordinates + epsilon), then
multiplies coordinates by learned scale parameters. Epsilon, here 1e-5, keeps
the denominator positive. RMS normalization controls each vector's scale
independently. Softmax serves a different purpose: producing normalized weights.

The FFN projects each column into F hidden coordinates. One projection gives u;
another gives gate. The nonlinear function *SiLU* is u/(1+exp(-u)). Multiplying
SiLU(u) by gate lets one computed feature modulate another. Gate entries are
general real-valued multipliers that can amplify, attenuate, or reverse a feature.
A final projection
returns to M coordinates so the update can be added to X. Here f labels a
hidden coordinate, and F=7.

Track just one hidden feature at one position. Its row in W1 reads a number
u from the normalized representation; its row in W3 reads a multiplier.
Their product after SiLU sets how far to move along column f of W2.
For example, take u=2 and compare three possible gate values:

*Predict before running:* For gate values 0, 0.5, and -0.5, will the update
vanish, follow the write direction, or oppose it? You only need the sign,
not the numerical value of SiLU(2).


In [ ]:
feature_u = 2.0
feature_gate = np.array([0., 0.5, -0.5])
feature_response = feature_u / (1 + np.exp(-feature_u)) * feature_gate
feature_write = np.array([0., 0., 2., 2.])
# The local axis n enumerates the three alternative gate values.
feature_updates = einsum(feature_write, feature_response, 'm, n -> m n')
feature_response, feature_updates


The responses are approximately 0, 0.881, and -0.881. The same write direction
therefore receives zero, positive, or negative contributions. This extends
the earlier detector-and-write construction: the response is now a smooth,
input-dependent product. Each hidden feature supplies its own write direction;
`m f, f t -> m t` adds all their contributions. The lab's bias-free gated network
uses this mechanism in place of the earlier hand-built threshold detector.

To predict the next token, a final matrix produces one score for each vocabulary
entry. These unnormalized scores are called *logits*. In logits[v,t], v labels
a vocabulary entry. Softmax over v turns the
last column into next-token probabilities. This is a different distribution
from attention weights, which were normalized over source positions s.


In [ ]:
def rms_columns(x, weight):
    # Original: np.mean(x*x, axis=0, keepdims=True)
    mean_square = reduce(x*x, 'm t -> 1 t', 'mean')
    return weight[:,None] * x / np.sqrt(mean_square + 1e-5)
X = E + delta_h
F = 7
W1 = rng.normal(scale=0.2, size=(F,M))
W3 = rng.normal(scale=0.2, size=(F,M))
W2 = rng.normal(scale=0.2, size=(M,F))
normalized = rms_columns(X, np.ones(M))
u = einsum(W1, normalized, 'f m, m t -> f t')
gate = einsum(W3, normalized, 'f m, m t -> f t')
hidden = (u / (1 + np.exp(-u))) * gate              # SiLU(u) * gate
X = X + einsum(W2, hidden, 'm f, f t -> m t')
vocabulary = ['small','blue','robot','moved','stopped','today']
classifier = rng.normal(scale=0.2, size=(len(vocabulary),M))
logits = einsum(classifier, rms_columns(X, np.ones(M)), 'v m, m t -> v t')
next_probabilities = softmax_columns(logits[:,-1:])


In [ ]:
# Rendering
matrix_picture(next_probabilities, 'Final position: scores become vocabulary probabilities', vocabulary, ['next'])


We now have the operations needed to assemble a complete model. Each full block
also normalizes before attention and applies positional rotations. The next
section connects these steps and repeats the blocks. The probabilities here
illustrate computation with random parameters; trained parameters give the
model its predictive ability. GPT-style models share the attention structure
with variations in normalization, positional encoding and feed-forward functions.

<a id="part-iv"></a>


## Part IV: Life of a Token


In [ ]:
# Rendering
journey('4')


### Connecting the complete model

We now connect these operations. A *transformer block* combines attention and
an FFN, each with a residual addition. Stacking blocks lets later blocks work
with representations that already contain information gathered by earlier ones.
Our model uses two blocks with the lab's Llama-style operations.

A *forward pass* is one evaluation of the model with fixed parameters. It maps
an input sequence to vocabulary scores. Repeatedly choosing the next token from
those scores and appending it is called *autoregressive generation*.

There are two very different loops. The *layer loop* refines representations
while keeping the number of positions unchanged. The *generation loop* chooses
one new token, appends it, and runs the model again on a longer sequence.


```text
text -> token IDs -> embedding columns
                     |
                     +-- block 0: normalize -> attention -> add residual
                     |            normalize -> gated feed-forward -> add residual
                     +-- block 1: same operations, DIFFERENT weights
                     |
                  final normalization -> vocabulary scores at every position
                     |
                  last column -> softmax -> choose token -> append to text
                                                           |
                                       run the whole model again
```


### One prompt, fixed parameters

We use a six-word vocabulary and split text on spaces. This is a complete
tokenizer for this restricted example. The lab uses its model's own tokenizer.
Inputs must be nonempty sequences of these vocabulary words. We omit special
tokens and stop generation after a specified number of new tokens.

The *prompt* is the initial text we ask the model to continue. The embedding
table and other parameters below are randomly initialized and fixed across calls.
The resulting text illustrates execution with an untrained model: tokenization,
lookup, positional rotation, two full transformer blocks, prediction, and
autoregressive generation. Training would give these parameters predictive value.


In [ ]:
model_rng = np.random.default_rng(2026)
words = ['small', 'blue', 'robot', 'moved', 'stopped', 'today']
word_to_id = {word: i for i, word in enumerate(words)}
def encode(text):
    return [word_to_id[word] for word in text.split()]
def decode(ids):
    return ' '.join(words[i] for i in ids)
model_dim, query_heads, kv_heads, head_dim, hidden_dim = 8, 4, 2, 2, 12
assert model_dim == query_heads * head_dim
def weight(*shape):
    return model_rng.normal(scale=0.2, size=shape)
model = {
    'embedding': weight(model_dim, len(words)),
    'layers': [dict(
        att_norm=np.ones(model_dim), ffn_norm=np.ones(model_dim),
        wq=weight(query_heads, head_dim, model_dim),
        wk=weight(kv_heads, head_dim, model_dim),
        wv=weight(kv_heads, head_dim, model_dim),
        wo=weight(model_dim, query_heads, head_dim),
        w1=weight(hidden_dim, model_dim),
        w3=weight(hidden_dim, model_dim),
        w2=weight(model_dim, hidden_dim)
    ) for layer in range(2)],
    'final_norm': np.ones(model_dim),
    'classifier': weight(len(words), model_dim)
}
prompt = 'small blue robot'
ids = encode(prompt)
assert decode(ids) == prompt
print('Text:', prompt)
print('Token IDs:', ids)


In [ ]:
# Rendering
matrix_picture(model['embedding'][:, ids], 'Embedding lookup: X[m,t]',
               [f'm{i}' for i in range(model_dim)], prompt.split())


### Position enters through query/key rotations

Position supplies another part of the context. RoPE
lets matching depend on where tokens occur: "robot followed child" and
"child followed robot" use the same words with different relationships.
Our earlier query/key maps supply content-dependent vectors. RoPE
rotates adjacent pairs of query/key coordinates by an angle determined by
position. The same position-dependent rotation is applied to queries and keys;
value vectors retain their original coordinates. A dot product between differently rotated vectors
therefore depends on their relative positions as well as their content.

For a pair (a,b), the rotated coordinates are (a*cos(theta)-b*sin(theta),
a*sin(theta)+b*cos(theta)). Pair i at position t uses angle
t / 10000**(2*i/D), matching the lab's basic RoPE convention. Production models
may use other frequency bases or scaling. Here we use the lab's base of 10000.

Consider one coordinate pair with one radian of rotation per position.
Put a query at position 2 and a key at position 1. Then move both one position
forward.

*Predict before running:* Does moving both positions change the dot product?
What if only the query moves? Think about the angle between the two arrows.


In [ ]:
def rotate_pair(vector, angle):
    c, s = np.cos(angle), np.sin(angle)
    # Original: np.array([[c, -s], [s, c]]) @ vector
    return einsum(np.array([[c, -s], [s, c]]), vector, 'i j, j -> i')
rope_query = np.array([1., 0.])
rope_key = np.array([1., 0.5])
rope_q2, rope_k1 = rotate_pair(rope_query, 2), rotate_pair(rope_key, 1)
rope_q3, rope_k2 = rotate_pair(rope_query, 3), rotate_pair(rope_key, 2)
# Original: [rope_q2 @ rope_k1, rope_q3 @ rope_k2, rope_q3 @ rope_k1]
rope_scores = np.array([einsum(rope_q2, rope_k1, 'd, d -> '),
                        einsum(rope_q3, rope_k2, 'd, d -> '),
                        einsum(rope_q3, rope_k1, 'd, d -> ')])
np.testing.assert_allclose(rope_scores[0], rope_scores[1])
rope_scores


In [ ]:
# Rendering
fig = Geometry(['Query 2, key 1', 'Query 3, key 2'], bound=1.5)
for panel, query, key in [(0, rope_q2, rope_k1), (1, rope_q3, rope_k2)]:
    fig.vector(panel, query, COLORS[0], 'query')
    fig.vector(panel, key, COLORS[1], 'key')
    fig.caption(panel, f'Dot product {rope_scores[panel]:.3f}; separation 1 position')
fig.show()


Both panels give approximately 0.961; moving only the query gives 0.039.
Moving both rotates the arrows equally, preserving their angle difference
and dot product. Moving only the query changes that difference.
In symbols, `R(t)q · R(s)k ` q · R(s-t)k=: a common rotation cancels in
the dot product. Each coordinate pair uses its own rotation rate, giving
several views of the same relative separation. Content still determines the
unrotated vectors q and k.


In [ ]:
def rotate_positions(z):
    # z[h,d,t]; reshape groups adjacent coordinates within each head and position.
    heads, width, steps = z.shape
    # Original: z.reshape(heads, width//2, 2, steps)
    pairs = rearrange(z, 'h (p c) t -> h p c t', c=2)
    angle = np.arange(steps)[None,:] / 10000**(
        np.arange(0, width, 2)[:,None] / width)
    cosine, sine = np.cos(angle), np.sin(angle)
    a, b = pairs[:,:,0,:], pairs[:,:,1,:]
    rotated = np.stack((a*cosine-b*sine, a*sine+b*cosine), axis=2)
    # Original: rotated.reshape(z.shape)
    return rearrange(rotated, 'h p c t -> h (p c) t')
# Rotation preserves lengths; position zero is unchanged.
probe = model_rng.normal(size=(2,4,5))
rotated_probe = rotate_positions(probe)
np.testing.assert_allclose(np.sum(probe**2,axis=1),
                           np.sum(rotated_probe**2,axis=1))
np.testing.assert_allclose(probe[:,:,0], rotated_probe[:,:,0])


### One block: communicate, then transform at each position

The block below contains both residual additions. Attention reads normalized
activations but adds its output to the original residual. The feed-forward
part normalizes the updated residual and adds another update. Normalization
acts on the branch that computes the update; the residual path carries x forward.

Read the equations using their indices: h is a query head, g a shared KV head,
d a head coordinate, m an embedding coordinate, f a hidden coordinate, s a
source position and t a receiving position. Query and KV heads can have different
counts; repeating each KV head establishes their correspondence. The local
axis r counts query heads sharing a KV head; combining (g r) gives h.

The complete block assembles operations we have already used. Here is the
correspondence to keep beside the code:

| Earlier operation | In transformer_block | Result axes |
|-------------------|----------------------|-------------|
| Control each vector's scale | rms_columns | m,t |
| Read queries, keys, and values | wq, wk, wv projections | h,d,t; g,d,s; g,d,s |
| Add position and share KV heads | rotate_positions; repeat | h,d,t; h,d,s |
| Compare, mask, and normalize over sources | score; causal; probability | h,s,t |
| Gather messages and write an update | context; wo projection | h,d,t then m,t |
| Read features and modulate them | w1, w3; SiLU(u) * gate | f,t |
| Write feature contributions | w2 projection | m,t |

The examples used small, readable coordinates; this model uses M=8, H=4,
G=2, D=2, and F=12. The axis meanings stay the same. Both updates return to
m,t so they can be added to the representation flowing through the block.


In [ ]:
def transformer_block(x, layer):
    n = rms_columns(x, layer['att_norm'])
    q = rotate_positions(einsum(layer['wq'], n, 'h d m, m t -> h d t'))
    k = rotate_positions(einsum(layer['wk'], n, 'g d m, m s -> g d s'))
    v = einsum(layer['wv'], n, 'g d m, m s -> g d s')

    heads, width, steps = q.shape
    # Original: head_map = np.arange(heads) // (heads // k.shape[0])
    #           k, v = k[head_map], v[head_map]
    shared = heads // k.shape[0]
    k = repeat(k, 'g d s -> (g r) d s', r=shared)  # h,d,s
    v = repeat(v, 'g d s -> (g r) d s', r=shared)
    score = einsum(k, q, 'h d s, h d t -> h s t') / np.sqrt(width)
    causal = np.arange(steps)[:,None] <= np.arange(steps)[None,:]
    score = np.where(causal[None,:,:], score, -np.inf)
    # Original: score - score.max(axis=1, keepdims=True)
    numerator = np.exp(score - reduce(score, 'h s t -> h 1 t', 'max'))
    # Original: numerator / numerator.sum(axis=1, keepdims=True)
    probability = numerator / reduce(numerator, 'h s t -> h 1 t', 'sum')
    context = einsum(v, probability, 'h d s, h s t -> h d t')
    x = x + einsum(layer['wo'], context, 'm h d, h d t -> m t')
    after_attention = x

    n = rms_columns(x, layer['ffn_norm'])
    u = einsum(layer['w1'], n, 'f m, m t -> f t')
    gate = einsum(layer['w3'], n, 'f m, m t -> f t')
    hidden = (u / (1 + np.exp(-u))) * gate          # SiLU(u) * gate
    x = x + einsum(layer['w2'], hidden, 'm f, f t -> m t')
    return x, after_attention, probability


Each layer computes all positions
together, with the causal mask preventing future information from entering
earlier positions. The layers themselves remain sequential: block 1 consumes
block 0's updated representations.


### The complete forward pass, and a trace we can inspect

The output column at position t predicts the token after position t. For
generation we use the last column. Each earlier column similarly predicts the
token following its own position.


In [ ]:
def transformer_forward(ids, model):
    x = model['embedding'][:, ids]
    trace = [('embedding', x.copy())]
    patterns = []
    for number, layer in enumerate(model['layers']):
        x, after_attention, probability = transformer_block(x, layer)
        trace.append((f'block {number}: attention residual', after_attention.copy()))
        trace.append((f'block {number}: feed-forward residual', x.copy()))
        patterns.append(probability)
    x = rms_columns(x, model['final_norm'])
    trace.append(('final normalization', x.copy()))
    logits = einsum(model['classifier'], x, 'v m, m t -> v t')
    return logits, trace, patterns
full_logits, trace, patterns = transformer_forward(ids, model)
for stage, activation in trace:
    print(f'{stage:36} shape={activation.shape}  last token={activation[:3,-1]} ...')
print('Vocabulary scores:', full_logits.shape)
# Each column shows the SAME token as it passes through successive stages.
trajectory = np.column_stack([activation[:,-1] for stage, activation in trace])
next_distribution = softmax_columns(full_logits[:,-1:])
selected = int(np.argmax(next_distribution[:,0]))
print('Greedy selection:', selected, words[selected])


In [ ]:
# Rendering
matrix_picture(trajectory, 'Follow robot through the complete model',
               [f'm{i}' for i in range(model_dim)],
               ['lookup','0 attn','0 FFN','1 attn','1 FFN','norm'])
matrix_picture(next_distribution, 'After the last block: next-token distribution',
               words, ['after robot'])


Notice that every intermediate residual has model_dim rows and one column per
input position: here eight rows and three columns. Attention changes
what information each column carries. The caller then chooses a vocabulary item
and appends it, lengthening the sequence by one position.

The vocabulary-score array has shape (V,T). Column t predicts the token after
position t using its available prefix. For generation, the final column predicts
the continuation beyond the input. Earlier columns correspond to positions
whose successors are already present. The causal mask makes all these
prefix-based predictions possible in one forward pass.


In [ ]:
# Rendering
candidate_picture(words,full_logits[:,-1],next_distribution[:,0],
                  'Complete model: candidates after small blue robot')


### Autoregressive generation: feed the result back in

We first use greedy selection to obtain a simple repeatable trace, then run
seeded sampling. Each generation step uses a newly computed distribution.
The model parameters stay fixed throughout both runs.

*Predict before running:* Extend the prompt from "small blue robot" to
"small blue robot moved". Will the logits at the robot position change?
Will the next-token distribution now come from the same position?


In [ ]:
def generate(text, new_tokens, model, sample=False, temperature=1.0, seed=326):
    ids = encode(text)
    history = []
    rng = np.random.default_rng(seed)
    for step in range(new_tokens):
        logits, _, _ = transformer_forward(ids, model)
        probability = token_probabilities(logits[:,-1],temperature)
        if sample:
            chosen = int(rng.choice(len(words),p=probability))
        else:
            chosen = int(np.argmax(probability))
        history.append((decode(ids), words[chosen], float(probability[chosen])))
        ids.append(chosen)
    return decode(ids), history
generated, history = generate(prompt, 4, model)
for prefix, chosen, confidence in history:
    print(f'{prefix!r} -> {chosen!r} (probability {confidence:.3f})')
print('Result with UNTRAINED weights:', generated)
# A causal model must give the same earlier predictions even when future
# tokens are appended. Test every prefix, across both complete layers.
long_ids = encode('small blue robot moved today')
long_logits, _, _ = transformer_forward(long_ids, model)
for length in range(1, len(long_ids)+1):
    prefix_logits, _, _ = transformer_forward(long_ids[:length], model)
    np.testing.assert_allclose(prefix_logits, long_logits[:,:length], atol=1e-12)
print('All complete-model prefix checks passed')


Robot's logits remain unchanged because the causal mask excludes the new
future token. The next-token distribution now comes from moved, the new
last position. The prefix checks above verify the first claim through both
complete transformer blocks.


In [ ]:
sampled_text, sampled_history = generate(prompt,4,model,sample=True,temperature=0.8)
for prefix,chosen,probability in sampled_history:
    print(f'{prefix!r} -> {chosen!r} (probability {probability:.3f})')
print('Sampled continuation with UNTRAINED weights:',sampled_text)


The choice rule changes how we use the model's probabilities. Both examples
stop after four generated tokens. A trained model with an end-of-sequence
token can additionally stop when that token is selected.

This recomputes the entire prefix at each generation step. It is intentionally
simple and corresponds to the lab's stateless full-prefix inference behavior.
Caching offers a more efficient implementation by reusing earlier work.
Our implementation batches positions inside each layer; the lab's baseline
loops through positions. Both must enforce the same information dependency.


### What would make the output meaningful?

Meaningful predictions require trained parameters together with their matching
model dimensions and tokenizer. *Inference* means evaluating that trained model
on an input. The lab supplies the trained parameters and tokenizer; our miniature
model exposes the same sequence of computations using small, inspectable arrays.


### What the notation lets us see

- Projection reduces embedding coordinates, independently at each position.
- Comparing queries and keys retains two position indices: all pairs.
- Gathering values removes the source-position index: weighted communication.
- The feed-forward block retains t throughout: each position is transformed independently.
- Heads are another free index until the output projection combines them.
- Lookup, masking, exponentiation, normalization and nonlinear activation
  complete the model around einsum's products and sums.

The entire transformer forward computation is now visible in this small
vocabulary: einsum, rearrangement, generalized reduction, elementwise
mathematics, and indexing. The same array language expresses both a single
geometric operation and the complete model that predicts the next token.

For self-study, change one ingredient at a time: an input column, a projection,
the mask, or a value vector. Predict which intermediate arrays will change before
running the notebook. Follow the information from its source to its effect on
the next-token prediction.


## Recap

A transformer repeatedly refines vector representations of tokens. Geometry
explains what each transformation does; named axes express how it acts across
coordinates, positions, and heads.

We learned:

1. how embeddings represent tokens as vectors, and learned projections read
   and write directions in those representations;
2. how attention compares queries with keys, turns scores into weights, and
   gathers values to transfer information between positions;
3. how feed-forward networks recognize features and write input-dependent
   updates using associations encoded in their parameters;
4. how normalization controls scale, residual additions carry representations
   forward, and positional rotations make comparisons sensitive to relative
   position;
5. how a causal mask restricts each position to its available prefix, while
   stacked blocks successively refine the information in that prefix;
6. how the final representation produces vocabulary logits and probabilities,
   and choosing and appending a token turns prediction into generation.

The central payoff is expressive power: the complete transformer forward
computation fits a small array vocabulary. `einsum` expresses products and
sums, `rearrange` exposes structure, `reduce` combines entries, and `repeat`
shares them. Elementwise mathematics and indexing complete the computation.
The surviving and reduced axes tell us what each expression means.

We can now follow a token from its initial embedding to a prediction, read
the corresponding tensor expressions, and implement the entire forward pass.
Trained parameters give this same computational structure its predictive
ability; our small examples make the structure visible.


## Attribution and further reading

Visual conventions and numeric-matrix primitives adapted from Grant Sanderson /
3Blue1Brown: [Transformers](https://www.3blue1brown.com/lessons/gpt/),
[Attention](https://www.3blue1brown.com/lessons/attention/), and
[How might LLMs store facts?](https://www.3blue1brown.com/lessons/mlp/).
The geometric diagrams are teaching adaptations computed from the examples
in this notebook. The matrix visual code draws on
[WeightMatrix, NumericEmbedding, EmbeddingArray and value_to_color](https://github.com/3b1b/videos/blob/31f8ffe785e860477a6051fde983c02c7d165bc2/_2024/transformers/helpers.py),
adapted into static SVG figures. The adapted visuals
and their rendering code are licensed under
[Creative Commons Attribution-NonCommercial-ShareAlike 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).
